In [1]:
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multioutput import ClassifierChain
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import make_scorer, f1_score
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report
from sklearn.metrics import hamming_loss, accuracy_score
from sklearn.metrics import confusion_matrix

from iterstrat.ml_stratifiers import MultilabelStratifiedKFold


import hvplot.pandas
import holoviews as hv
hv.extension('bokeh')



In [2]:
df_train = pd.read_csv('emotions/emotions_train.csv')

In [3]:
df_test = pd.read_csv('emotions/emotions_test.csv')

In [4]:
df_train.head()

,Mean_Acc1298_Mean_Mem40_Centroid,Mean_Acc1298_Mean_Mem40_Rolloff,Mean_Acc1298_Mean_Mem40_Flux,Mean_Acc1298_Mean_Mem40_MFCC_0,Mean_Acc1298_Mean_Mem40_MFCC_1,Mean_Acc1298_Mean_Mem40_MFCC_2,Mean_Acc1298_Mean_Mem40_MFCC_3,Mean_Acc1298_Mean_Mem40_MFCC_4,Mean_Acc1298_Mean_Mem40_MFCC_5,Mean_Acc1298_Mean_Mem40_MFCC_6,...,BH_HighLowRatio,BHSUM1,BHSUM2,BHSUM3,amazed-suprised,happy-pleased,relaxing-calm,quiet-still,sad-lonely,angry-aggresive
0,0.034741,0.089665,0.091225,-73.302422,6.215179,0.615074,2.037160,0.804065,1.301409,0.558576,...,2.0,0.245457,0.105065,0.405399,0,1,1,0,0,0
1,0.081374,0.272747,0.085733,-62.584437,3.183163,-0.218145,0.163038,0.620251,0.458514,0.041426,...,2.0,0.343547,0.276366,0.710924,1,0,0,0,0,1
2,0.110545,0.273567,0.084410,-65.235325,2.794964,0.639047,1.281297,0.757896,0.489412,0.627636,...,3.0,0.188693,0.045941,0.457372,0,1,0,0,0,1
3,0.042481,0.199281,0.093447,-80.305152,5.824409,0.648848,1.754870,1.495532,0.739909,0.809644,...,2.0,0.102839,0.241934,0.351009,0,0,1,0,0,0
4,0.074550,0.140880,0.079789,-93.697749,5.543229,1.064262,0.899152,0.890336,0.702328,0.490685,...,2.0,0.195196,0.310801,0.683817,0,0,0,1,0,0


In [5]:
df_test.head()

,Mean_Acc1298_Mean_Mem40_Centroid,Mean_Acc1298_Mean_Mem40_Rolloff,Mean_Acc1298_Mean_Mem40_Flux,Mean_Acc1298_Mean_Mem40_MFCC_0,Mean_Acc1298_Mean_Mem40_MFCC_1,Mean_Acc1298_Mean_Mem40_MFCC_2,Mean_Acc1298_Mean_Mem40_MFCC_3,Mean_Acc1298_Mean_Mem40_MFCC_4,Mean_Acc1298_Mean_Mem40_MFCC_5,Mean_Acc1298_Mean_Mem40_MFCC_6,...,BH_HighLowRatio,BHSUM1,BHSUM2,BHSUM3,amazed-suprised,happy-pleased,relaxing-calm,quiet-still,sad-lonely,angry-aggresive
0,0.036299,0.064986,0.082104,-72.710462,7.920220,0.134279,2.546373,0.671063,1.589821,0.576485,...,2.0,0.095982,0.520006,0.677943,0,0,1,1,1,0
1,0.161218,0.467820,0.096983,-71.298043,1.176349,1.871744,1.097346,0.641059,0.372797,0.991050,...,2.0,0.752210,0.576382,1.477141,1,0,0,0,0,1
2,0.115987,0.336879,0.079068,-64.570939,2.339044,0.714859,1.792451,0.611347,0.287022,0.772846,...,2.0,0.488375,0.004603,1.147727,0,0,0,0,1,0
3,0.086016,0.141845,0.081554,-81.141092,6.714252,-1.338896,1.326248,0.340032,1.290664,0.337209,...,2.0,0.430059,0.102757,1.276632,0,1,1,0,0,0
4,0.063232,0.140621,0.082097,-66.596131,5.594724,0.350716,1.023655,0.439544,0.855564,0.414784,...,2.0,1.788567,0.032760,3.076057,0,0,0,0,1,0


In [6]:
emo_cols = [
    'amazed-suprised',
    'happy-pleased',
    'relaxing-calm',
    'quiet-still',
    'sad-lonely',
    'angry-aggresive'
]

X_train = df_train.drop(columns=emo_cols)
y_train = df_train[emo_cols]

In [7]:
# 1. define model
rf = OneVsRestClassifier(
    RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1)
)

# 2. define multilabel stratified CV
cv = MultilabelStratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 3. define metrics
scoring = {
    'f1_macro': make_scorer(
        f1_score,
        average='macro',
        zero_division=0
    ),
    'f1_micro': make_scorer(
        f1_score,
        average='micro',
        zero_division=0
    )
}

#4. run multilabel stratified CV
cv_results = cross_validate(
    rf,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

# 5. print metric scores
print(
    'Macro F1:',
    cv_results['test_f1_macro'].mean(),
    '+/-',
    cv_results['test_f1_macro'].std()
)

print(
    'Micro F1:',
    cv_results['test_f1_micro'].mean(),
    '+/-',
    cv_results['test_f1_micro'].std()
)

Macro F1: 0.6152636260086244 +/- 0.01760882846810418
Micro F1: 0.6568018863866204 +/- 0.01819697129909844


In [8]:
# 1. define model
rf = OneVsRestClassifier(
    RandomForestClassifier(
        n_estimators=300,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1)
)

# 2. define multilabel stratified CV
cv = MultilabelStratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 3. define metrics
scoring = {
    'f1_macro': make_scorer(
        f1_score,
        average='macro',
        zero_division=0
    ),
    'f1_micro': make_scorer(
        f1_score,
        average='micro',
        zero_division=0
    )
}

#4. run multilabel stratified CV
cv_results = cross_validate(
    rf,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

# 5. print metric scores
print(
    'Macro F1:',
    cv_results['test_f1_macro'].mean(),
    '+/-',
    cv_results['test_f1_macro'].std()
)

print(
    'Micro F1:',
    cv_results['test_f1_micro'].mean(),
    '+/-',
    cv_results['test_f1_micro'].std()
)

Macro F1: 0.5958675871006511 +/- 0.018072644462011952
Micro F1: 0.637049893652848 +/- 0.019340722720160564


In [9]:
# 5. classification report
y_pred_cv = cross_val_predict(
    rf,
    X_train,
    y_train,
    cv=cv,
    method='predict'
)

print(
    'Hamming Loss:',
    hamming_loss(y_train, y_pred_cv)
)

print(
    'Subset Accuracy:',
    accuracy_score(y_train, y_pred_cv)
)

hamming_per_label = {}

for i, emotion in enumerate(emo_cols):
    hamming_per_label[emotion] = hamming_loss(
        y_train.iloc[:, i],
        y_pred_cv[:, i]
    )

report_df = pd.DataFrame(
    classification_report(
        y_train,
        y_pred_cv,
        target_names=emo_cols,
        zero_division=0,
        output_dict=True
    )
).T

report_df['hamming_loss'] = pd.Series(hamming_per_label)

report_df

Hamming Loss: 0.1918158567774936
Subset Accuracy: 0.2864450127877238


,precision,recall,f1-score,support,hamming_loss
amazed-suprised,0.712644,0.521008,0.601942,119.0,0.209719
happy-pleased,0.518519,0.130841,0.208955,107.0,0.271100
relaxing-calm,0.758427,0.803571,0.780347,168.0,0.194373
quiet-still,0.864865,0.719101,0.785276,89.0,0.089514
sad-lonely,0.826923,0.452632,0.585034,95.0,0.156010
angry-aggresive,0.681416,0.587786,0.631148,131.0,0.230179
micro avg,0.743879,0.557123,0.637097,709.0,NaN
macro avg,0.727132,0.535823,0.598784,709.0,NaN
weighted avg,0.722846,0.557123,0.611052,709.0,NaN
samples avg,0.622762,0.555413,0.560614,709.0,NaN


In [10]:
# 6. confusion matrices
for i, emotion in enumerate(emo_cols):

    cm = confusion_matrix(
        y_train.iloc[:, i],
        y_pred_cv[:, i]
    )

    cm_df = pd.DataFrame(
        cm,
        index=['No', 'Yes'],
        columns=['No', 'Yes']
    )

    cm_long = (
        cm_df
        .rename_axis('Actual')
        .reset_index()
        .melt(
            id_vars='Actual',
            var_name='Predicted',
            value_name='Count'
        )
    )

    display(
        cm_long.hvplot.heatmap(
            x='Predicted',
            y='Actual',
            C='Count',
            title=f'Confusion Matrix — {emotion}',
            width=500,
            height=400,
            colorbar=True
        )
    )

:HeatMap   [Predicted,Actual]   (Count)

:HeatMap   [Predicted,Actual]   (Count)

:HeatMap   [Predicted,Actual]   (Count)

:HeatMap   [Predicted,Actual]   (Count)

:HeatMap   [Predicted,Actual]   (Count)

:HeatMap   [Predicted,Actual]   (Count)

#### RF classifier chain

In [12]:
rf_chain = ClassifierChain(
    RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),
    order='random',
    random_state=42
)

# 2. define multilabel stratified CV
cv = MultilabelStratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 3. define metrics
scoring = {
    'f1_macro': make_scorer(
        f1_score,
        average='macro',
        zero_division=0
    ),
    'f1_micro': make_scorer(
        f1_score,
        average='micro',
        zero_division=0
    )
}

#4. run multilabel stratified CV
cv_results = cross_validate(
    rf_chain,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

# 5. print metric scores
print(
    'Macro F1:',
    cv_results['test_f1_macro'].mean(),
    '+/-',
    cv_results['test_f1_macro'].std()
)

print(
    'Micro F1:',
    cv_results['test_f1_micro'].mean(),
    '+/-',
    cv_results['test_f1_micro'].std()
)

Macro F1: 0.6287550515776796 +/- 0.015796262222149204
Micro F1: 0.6760472340388417 +/- 0.01885921804543659


In [13]:
# 5. classification report
y_pred_cv = cross_val_predict(
    rf_chain,
    X_train,
    y_train,
    cv=cv,
    method='predict'
)

print(
    'Hamming Loss:',
    hamming_loss(y_train, y_pred_cv)
)

print(
    'Subset Accuracy:',
    accuracy_score(y_train, y_pred_cv)
)

hamming_per_label = {}

for i, emotion in enumerate(emo_cols):
    hamming_per_label[emotion] = hamming_loss(
        y_train.iloc[:, i],
        y_pred_cv[:, i]
    )

report_df = pd.DataFrame(
    classification_report(
        y_train,
        y_pred_cv,
        target_names=emo_cols,
        zero_division=0,
        output_dict=True
    )
).T

report_df['hamming_loss'] = pd.Series(hamming_per_label)

report_df

Hamming Loss: 0.17902813299232737
Subset Accuracy: 0.3273657289002558


,precision,recall,f1-score,support,hamming_loss
amazed-suprised,0.726316,0.579832,0.644860,119.0,0.194373
happy-pleased,0.538462,0.130841,0.210526,107.0,0.268542
relaxing-calm,0.763158,0.863095,0.810056,168.0,0.173913
quiet-still,0.841463,0.775281,0.807018,89.0,0.084399
sad-lonely,0.796610,0.494737,0.610390,95.0,0.153453
angry-aggresive,0.696296,0.717557,0.706767,131.0,0.199488
micro avg,0.746167,0.617772,0.675926,709.0,NaN
macro avg,0.727051,0.593557,0.631603,709.0,NaN
weighted avg,0.725022,0.617772,0.645631,709.0,NaN
samples avg,0.676471,0.618500,0.616709,709.0,NaN


In [14]:
# 6. confusion matrices
for i, emotion in enumerate(emo_cols):

    cm = confusion_matrix(
        y_train.iloc[:, i],
        y_pred_cv[:, i]
    )

    cm_df = pd.DataFrame(
        cm,
        index=['No', 'Yes'],
        columns=['No', 'Yes']
    )

    cm_long = (
        cm_df
        .rename_axis('Actual')
        .reset_index()
        .melt(
            id_vars='Actual',
            var_name='Predicted',
            value_name='Count'
        )
    )

    display(
        cm_long.hvplot.heatmap(
            x='Predicted',
            y='Actual',
            C='Count',
            title=f'Confusion Matrix — {emotion}',
            width=500,
            height=400,
            colorbar=True
        )
    )

:HeatMap   [Predicted,Actual]   (Count)

:HeatMap   [Predicted,Actual]   (Count)

:HeatMap   [Predicted,Actual]   (Count)

:HeatMap   [Predicted,Actual]   (Count)

:HeatMap   [Predicted,Actual]   (Count)

:HeatMap   [Predicted,Actual]   (Count)

In [15]:
# 7. fit train data with chosen model
rf_chain.fit(X_train, y_train)

ClassifierChain(base_estimator=RandomForestClassifier(n_estimators=300,
                                                      n_jobs=-1,
                                                      random_state=42),
                order='random', random_state=42)

In [21]:
# 8. apply to test data
X_test = df_test.drop(columns=emo_cols)
y_test = df_test[emo_cols]

y_pred_test = rf_chain.predict(X_test)


# overall metrics
test_macro_f1 = f1_score(
    y_test,
    y_pred_test,
    average='macro',
    zero_division=0
)


test_micro_f1 = f1_score(
    y_test,
    y_pred_test,
    average='micro',
    zero_division=0
)

test_hamming = hamming_loss(
    y_test,
    y_pred_test
)

print('Test Macro F1:', test_macro_f1)
print('Test Micro F1:', test_micro_f1)
print('Test Hamming Loss:', test_hamming)



Test Macro F1: 0.6608703230016731
Test Micro F1: 0.6839945280437757
Test Hamming Loss: 0.1905940594059406


In [25]:
test_report_df = pd.DataFrame(
    classification_report(
        y_test,
        y_pred_test,
        target_names=emo_cols,
        zero_division=from sklearn.metrics import confusion_matrix

emotion = 'quiet-still'

i = emo_cols.index(emotion)

cm = confusion_matrix(
    y_test[emotion],
    y_pred_test[:, i]
)

print(cm)from sklearn.metrics import confusion_matrix

emotion = 'quiet-still'

i = emo_cols.index(emotion)

cm = confusion_matrix(
    y_test[emotion],
    y_pred_test[:, i]
)

print(cm)0,
        output_dict=True
    )
).T

test_report_df

,precision,recall,f1-score,support
amazed-suprised,0.690476,0.537037,0.604167,54.0
happy-pleased,0.782609,0.305085,0.439024,59.0
relaxing-calm,0.692982,0.822917,0.752381,96.0
quiet-still,0.762712,0.762712,0.762712,59.0
sad-lonely,0.888889,0.547945,0.677966,73.0
angry-aggresive,0.795918,0.672414,0.728972,58.0
micro avg,0.753012,0.626566,0.683995,399.0
macro avg,0.768931,0.608018,0.660870,399.0
weighted avg,0.767013,0.626566,0.670496,399.0
samples avg,0.658416,0.618812,0.613366,399.0


In [30]:
from sklearn.metrics import confusion_matrix

emotion = 'quiet-still'

i = emo_cols.index(emotion)

cm = confusion_matrix(
    y_test[emotion],
    y_pred_test[:, i]
)

print(cm)

[[129  14]
 [ 14  45]]
